# World Bank — Chile climate projects → city actions (world-bank-projects, v2)

Standalone review notebook for the **World Bank** source: pull mitigation projects from the World Bank Projects API, keep the **Chile** slice, standardise the fields, and map each project to the city action list at the sector/theme grain. Awards/projects layer, multilateral level, intermediated access. Writes two committed `data/` outputs: the Chile project dataset and the project→action crosswalk. The only external reference is the city action list; everything else is in this folder. Mediums/lows/none **await human adjudication**.

In [ ]:
%matplotlib inline
import os, requests, pandas as pd
ACTIONS='../../../../cl-ssg/cl-ssg-projects/releases/v1/data/derived/actions_profiled.csv'   # the city action list (the one allowed cross-review reference)
assert os.path.exists(ACTIONS), ACTIONS
acts=pd.read_csv(ACTIONS); acts['id']=acts[acts.columns[0]]
acat={r['id']:(str(r['action_name']).strip(), r['gpc_reference_number']) for _,r in acts.iterrows()}
len(acat)

In [ ]:
# Extract: World Bank Projects API (mitigation query), paginated, then keep Chile
base='https://search.worldbank.org/api/v2/projects?format=json&rows=100&os={}&qterm=mitigation'
allp=[]; o=0
while True:
    d=requests.get(base.format(o), timeout=60).json(); p=d.get('projects',{})
    if not p: break
    allp+=list(p.values())
    if len(p)<100: break
    o+=100
raw=pd.DataFrame(allp)
raw=raw[raw['countryname'].astype(str).str.contains('Chile', case=False, na=False)]
print('Chile World Bank mitigation projects:', len(raw))
raw.head()

In [ ]:
# Standardise to a tidy table
def to_num(s):
    try: return float(str(s).replace(',',''))
    except: return None
chile=pd.DataFrame({
  'source':'World Bank',
  'project_id':raw['id'],
  'project_name':raw['project_name'],
  'country_name':raw['countryname'],
  'region':raw.get('regionname'),
  'approval_date':raw.get('boardapprovaldate'),
  'closing_date':raw.get('closingdate'),
  'status':raw.get('status'),
  'total_commitment_amount':raw['totalcommamt'].apply(to_num) if 'totalcommamt' in raw else None,
  'sector':raw.get('sector1'),
  'theme':raw.get('theme1'),
  'instrument_type':raw.get('lendinginstr'),
  'url':raw.get('url'),
})
chile.head()

In [ ]:
# Map each project to a city action (sector/theme/name text, link-don't-attribute)
KW=[
  (['electric bus','e-bus','zero-emission bus','bus rapid','brt'], 'c40_0023','high','Electric/zero-emission buses -> adopt zero-emission bus fleets.'),
  (['e-mobility','electromobility','electric vehicle',' ev ','zero-emission transport'], 'c40_0023','medium','Electromobility / EV fleets -> zero-emission fleets (transport).'),
  (['solar','photovoltaic',' pv '], 'icare_0012','medium','Solar generation -> solar on public assets (utility-scale only a partial fit).'),
  (['energy efficiency','retrofit','thermal','insulation'], 'c40_0016','medium','Energy efficiency / retrofit -> building energy-efficiency retrofit.'),
  (['recycl','circular economy','solid waste','waste management'], 'c40_0037','medium','Recycling / waste management -> segregated collection of recyclables.'),
  (['compost','organic waste'], 'icare_0064','medium','Organic waste / compost -> organic waste management strategy.'),
  (['redd','deforestation','native forest','sustainable forest'], 'ipcc_0052','medium','REDD+ / avoided deforestation -> reduce deforestation and degradation.'),
  (['reforest','afforest','forest restoration'], 'ipcc_0053','medium','Afforestation / reforestation -> forest restoration.'),
  (['wetland','peatland'], 'ipcc_0060','medium','Wetland protection -> protect & restore wetlands.'),
  (['green space','urban park','green area','urban green'], 'c40_0042','medium','Urban green space -> expand urban & peri-urban green spaces.'),
  (['biogas','methane capture'], 'icare_0031','medium','Wastewater biogas -> capture & use biogas at treatment plants.'),
  (['green hydrogen','hydrogen'], 'icare_0078','low','Green hydrogen -> decarbonize industrial feedstocks (loose).'),
]
def map_action(text):
    t=' '+str(text).lower()+' '
    for kws,aid,conf,rat in KW:
        if any(k in t for k in kws): return aid,conf,rat
    return '', 'none', 'No specific mitigation intervention keyword; broad or out-of-scope project.'

TEXTCOLS=['project_name', 'sector', 'theme']
def row_text(r): return ' '.join(str(r.get(c,'')) for c in TEXTCOLS)
chile=chile.copy()
chile['_text']=chile.apply(row_text, axis=1)
m=chile['_text'].apply(map_action)
chile['action_id']=[x[0] for x in m]; chile['mapping_confidence']=[x[1] for x in m]; chile['rationale']=[x[2] for x in m]
chile['action_name']=chile['action_id'].map(lambda a: acat.get(a,('',''))[0] if a else '')
chile[['project_id','project_name','action_id','mapping_confidence']].head(50)

In [ ]:
# Validate, export the PROJECT DATASET + the crosswalk, coverage chart
for a in chile.loc[chile.action_id!='','action_id'].unique(): assert a in acat, a
os.makedirs('data', exist_ok=True)
chile.drop(columns=['_text']).to_csv('data/wb_chile_projects.csv', index=False)
chile[['project_id','project_name']+[c for c in TEXTCOLS if c!='project_name']+['action_id','action_name','mapping_confidence','rationale']].to_csv('data/wb_chile_to_actions.csv', index=False)
cov=chile['mapping_confidence'].value_counts().reindex(['high','medium','low','none']).fillna(0).astype(int)
mapped=int(cov[['high','medium','low']].sum())
print(f'Chile projects={len(chile)} | mapped={mapped} | none={int(cov["none"])}')
print('actions reached:', sorted(chile.loc[chile.action_id!="","action_id"].unique()))
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(6,2.6)); cov.plot.bar(ax=ax,color=['#1a7a3a','#6bbf59','#d9b54a','#bbbbbb'])
ax.set_title('World Bank Chile projects by mapping confidence'); ax.set_ylabel('projects')
fig.tight_layout(); fig

In [ ]:
# Mapper self-tests (independent of the data pull)
assert map_action('Santiago Electromobility and Clean Bus Transport Transportation Urban transport')[0]=='c40_0023'
assert map_action('Chile Solar Energy and Storage Program Energy Solar energy')[0]=='icare_0012'
assert map_action('Sustainable Cities and Climate Resilience Public Administration Climate change')[1]=='none'  # generic/broad or adaptation -> none
print('self-tests passed')

## Findings

- World Bank Chile climate operations are few (Chile is high-income, with limited Bank lending) and carry broad `theme1`/`sector1` labels, so expect a high `none` share: energy and transport operations map, broad policy or resilience operations do not.
- Adaptation is out of scope (mitigation-only action list); resilience and water operations map to `none` by design.
- Sovereign lending, so every row is intermediated; a city benefits as a sub-borrower. Link-don't-attribute; medium/low/none await human adjudication.